In [ ]:
## PROBA_RF
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

# ---------- 1) Load test ----------
test_gdf = gpd.read_file("test.geojson")

# ---------- 2) Feature engineering (IDENTIQUE au training) ----------
def add_geometry_features(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    if gdf.crs and gdf.crs.is_geographic:
        gdf = gdf.to_crs("EPSG:3857")

    gdf = gdf[gdf.geometry.notnull() & gdf.is_valid]
    geom = gdf.geometry
    out = pd.DataFrame(index=gdf.index)

    out["area"] = geom.area
    out["perimeter"] = geom.length

    bounds = geom.bounds
    out["bbox_w"] = bounds["maxx"] - bounds["minx"]
    out["bbox_h"] = bounds["maxy"] - bounds["miny"]
    out["bbox_aspect"] = out["bbox_w"] / (out["bbox_h"] + 1e-9)

    out["compactness"] = 4 * np.pi * out["area"] / ((out["perimeter"] ** 2) + 1e-9)
    hull_area = geom.convex_hull.area
    out["convexity_area_ratio"] = out["area"] / (hull_area + 1e-9)

    def n_vertices(g):
        try:
            return len(g.exterior.coords)
        except Exception:
            return 0

    out["n_vertices"] = geom.apply(n_vertices)
    return out


def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for k in range(1, 6):
        r = out[f"img_red_mean_date{k}"]
        g = out[f"img_green_mean_date{k}"]
        b = out[f"img_blue_mean_date{k}"]
        s = r + g + b + 1e-9
        out[f"bright_mean_date{k}"] = s / 3.0
        out[f"r_ratio_date{k}"] = r / s
        out[f"g_ratio_date{k}"] = g / s
        out[f"b_ratio_date{k}"] = b / s

    for c in ["red", "green", "blue"]:
        out[f"{c}_mean_delta_1_5"] = out[f"img_{c}_mean_date5"] - out[f"img_{c}_mean_date1"]
        out[f"{c}_std_delta_1_5"]  = out[f"img_{c}_std_date5"]  - out[f"img_{c}_std_date1"]

    for c in ["red", "green", "blue"]:
        means = np.vstack([out[f"img_{c}_mean_date{k}"].values for k in range(1, 6)]).T
        out[f"{c}_mean_volatility"] = means.std(axis=1)

    status_cols = [f"change_status_date{i}" for i in range(0, 5)]
    statuses = out[status_cols].astype(str)

    out["status_n_unique"] = statuses.nunique(axis=1)

    def count_transitions(row):
        seq = row.values
        return sum(seq[i] != seq[i-1] for i in range(1, len(seq)))

    out["status_transitions"] = statuses.apply(count_transitions, axis=1)
    return out


def extract_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    date_cols = []

    for col in df.columns:
        v = df[col].iloc[0]
        if isinstance(v, str) and ("-" in v or "/" in v):
            date_cols.append(col)

    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
        df[f"{col}_year"] = parsed.dt.year
        df[f"{col}_month"] = parsed.dt.month
        df[f"{col}_day"] = parsed.dt.day

    return df


# ---------- 3) Rebuild test features EXACTEMENT comme avant ----------
test_X = test_gdf.copy()

test_geom = add_geometry_features(test_gdf.loc[test_X.index])
test_feat = pd.concat([test_X.drop(columns=["geometry"]), test_geom], axis=1)

test_feat = add_temporal_features(test_feat)
test_feat = extract_date_features(test_feat)

# gérer les inf
test_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

# ---------- 4) Load model ----------
print("Loading saved model...")
best_model = joblib.load("model_rf.joblib")

# ---------- 5) Predict probabilities ----------
proba = best_model.predict_proba(test_feat)

# récupérer les labels dans l’ordre du modèle
class_labels = best_model.classes_

# ---------- 6) Save submission ----------
sub = pd.DataFrame(proba, columns=[f"proba_class_{c}" for c in class_labels])
sub.insert(0, "Id", np.arange(0, len(proba)))

sub.to_csv("PROBA_rf.csv", index=False)
print("Submission written: PROBA_rf.csv")



Loading saved model...
Submission written: PROBA_rf.csv


In [ ]:
## PROBA_BOOST
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

# ---------- 1) Load test ----------
test_gdf = gpd.read_file("test.geojson")

# ---------- 2) Feature engineering (IDENTIQUE au training) ----------
def add_geometry_features(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    if gdf.crs and gdf.crs.is_geographic:
        gdf = gdf.to_crs("EPSG:3857")

    gdf = gdf[gdf.geometry.notnull() & gdf.is_valid]
    geom = gdf.geometry
    out = pd.DataFrame(index=gdf.index)

    out["area"] = geom.area
    out["perimeter"] = geom.length

    bounds = geom.bounds
    out["bbox_w"] = bounds["maxx"] - bounds["minx"]
    out["bbox_h"] = bounds["maxy"] - bounds["miny"]
    out["bbox_aspect"] = out["bbox_w"] / (out["bbox_h"] + 1e-9)

    out["compactness"] = 4 * np.pi * out["area"] / ((out["perimeter"] ** 2) + 1e-9)
    hull_area = geom.convex_hull.area
    out["convexity_area_ratio"] = out["area"] / (hull_area + 1e-9)

    def n_vertices(g):
        try:
            return len(g.exterior.coords)
        except Exception:
            return 0

    out["n_vertices"] = geom.apply(n_vertices)
    return out


def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for k in range(1, 6):
        r = out[f"img_red_mean_date{k}"]
        g = out[f"img_green_mean_date{k}"]
        b = out[f"img_blue_mean_date{k}"]
        s = r + g + b + 1e-9
        out[f"bright_mean_date{k}"] = s / 3.0
        out[f"r_ratio_date{k}"] = r / s
        out[f"g_ratio_date{k}"] = g / s
        out[f"b_ratio_date{k}"] = b / s

    for c in ["red", "green", "blue"]:
        out[f"{c}_mean_delta_1_5"] = out[f"img_{c}_mean_date5"] - out[f"img_{c}_mean_date1"]
        out[f"{c}_std_delta_1_5"]  = out[f"img_{c}_std_date5"]  - out[f"img_{c}_std_date1"]

    for c in ["red", "green", "blue"]:
        means = np.vstack([out[f"img_{c}_mean_date{k}"].values for k in range(1, 6)]).T
        out[f"{c}_mean_volatility"] = means.std(axis=1)

    status_cols = [f"change_status_date{i}" for i in range(0, 5)]
    statuses = out[status_cols].astype(str)

    out["status_n_unique"] = statuses.nunique(axis=1)

    def count_transitions(row):
        seq = row.values
        return sum(seq[i] != seq[i-1] for i in range(1, len(seq)))

    out["status_transitions"] = statuses.apply(count_transitions, axis=1)
    return out


def extract_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    date_cols = []

    for col in df.columns:
        v = df[col].iloc[0]
        if isinstance(v, str) and ("-" in v or "/" in v):
            date_cols.append(col)

    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
        df[f"{col}_year"] = parsed.dt.year
        df[f"{col}_month"] = parsed.dt.month
        df[f"{col}_day"] = parsed.dt.day

    return df


# ---------- 3) Rebuild test features EXACTEMENT comme avant ----------
test_X = test_gdf.copy()

test_geom = add_geometry_features(test_gdf.loc[test_X.index])
test_feat = pd.concat([test_X.drop(columns=["geometry"]), test_geom], axis=1)

test_feat = add_temporal_features(test_feat)
test_feat = extract_date_features(test_feat)

# gérer les inf
test_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

# ---------- 4) Load model ----------
print("Loading saved model...")
best_model = joblib.load("model_boost.joblib")

# ---------- 5) Predict probabilities ----------
proba = best_model.predict_proba(test_feat)

# récupérer les labels dans l’ordre du modèle
class_labels = best_model.classes_

# ---------- 6) Save submission ----------
sub = pd.DataFrame(proba, columns=[f"proba_class_{c}" for c in class_labels])
sub.insert(0, "Id", np.arange(0, len(proba)))

sub.to_csv("PROBA_boost.csv", index=False)
print("Submission written: PROBA_boost.csv")



Loading saved model...
Submission written: PROBA_boost.csv


In [ ]:
## PROBA_RF_PCA
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib

# ---------- 1) Load test ----------
test_gdf = gpd.read_file("test.geojson")

# ---------- 2) Feature engineering (IDENTIQUE au training) ----------
def add_geometry_features(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    if gdf.crs and gdf.crs.is_geographic:
        gdf = gdf.to_crs("EPSG:3857")

    gdf = gdf[gdf.geometry.notnull() & gdf.is_valid]
    geom = gdf.geometry
    out = pd.DataFrame(index=gdf.index)

    out["area"] = geom.area
    out["perimeter"] = geom.length

    bounds = geom.bounds
    out["bbox_w"] = bounds["maxx"] - bounds["minx"]
    out["bbox_h"] = bounds["maxy"] - bounds["miny"]
    out["bbox_aspect"] = out["bbox_w"] / (out["bbox_h"] + 1e-9)

    out["compactness"] = 4 * np.pi * out["area"] / ((out["perimeter"] ** 2) + 1e-9)
    hull_area = geom.convex_hull.area
    out["convexity_area_ratio"] = out["area"] / (hull_area + 1e-9)

    def n_vertices(g):
        try:
            return len(g.exterior.coords)
        except Exception:
            return 0

    out["n_vertices"] = geom.apply(n_vertices)
    return out


def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for k in range(1, 6):
        r = out[f"img_red_mean_date{k}"]
        g = out[f"img_green_mean_date{k}"]
        b = out[f"img_blue_mean_date{k}"]
        s = r + g + b + 1e-9
        out[f"bright_mean_date{k}"] = s / 3.0
        out[f"r_ratio_date{k}"] = r / s
        out[f"g_ratio_date{k}"] = g / s
        out[f"b_ratio_date{k}"] = b / s

    for c in ["red", "green", "blue"]:
        out[f"{c}_mean_delta_1_5"] = out[f"img_{c}_mean_date5"] - out[f"img_{c}_mean_date1"]
        out[f"{c}_std_delta_1_5"]  = out[f"img_{c}_std_date5"]  - out[f"img_{c}_std_date1"]

    for c in ["red", "green", "blue"]:
        means = np.vstack([out[f"img_{c}_mean_date{k}"].values for k in range(1, 6)]).T
        out[f"{c}_mean_volatility"] = means.std(axis=1)

    status_cols = [f"change_status_date{i}" for i in range(0, 5)]
    statuses = out[status_cols].astype(str)

    out["status_n_unique"] = statuses.nunique(axis=1)

    def count_transitions(row):
        seq = row.values
        return sum(seq[i] != seq[i-1] for i in range(1, len(seq)))

    out["status_transitions"] = statuses.apply(count_transitions, axis=1)
    return out


def extract_date_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    date_cols = []

    for col in df.columns:
        v = df[col].iloc[0]
        if isinstance(v, str) and ("-" in v or "/" in v):
            date_cols.append(col)

    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
        df[f"{col}_year"] = parsed.dt.year
        df[f"{col}_month"] = parsed.dt.month
        df[f"{col}_day"] = parsed.dt.day

    return df

def to_dense(X):
    return X.toarray() if hasattr(X, 'toarray') else X


# ---------- 3) Rebuild test features EXACTEMENT comme avant ----------
test_X = test_gdf.copy()

test_geom = add_geometry_features(test_gdf.loc[test_X.index])
test_feat = pd.concat([test_X.drop(columns=["geometry"]), test_geom], axis=1)

test_feat = add_temporal_features(test_feat)
test_feat = extract_date_features(test_feat)

# gérer les inf
test_feat.replace([np.inf, -np.inf], np.nan, inplace=True)

# ---------- 4) Load model ----------
print("Loading saved model...")
best_model = joblib.load("model_rf_pca.joblib")

# ---------- 5) Predict probabilities ----------
proba = best_model.predict_proba(test_feat)

# récupérer les labels dans l’ordre du modèle
class_labels = best_model.classes_

# ---------- 6) Save submission ----------
sub = pd.DataFrame(proba, columns=[f"proba_class_{c}" for c in class_labels])
sub.insert(0, "Id", np.arange(0, len(proba)))

sub.to_csv("PROBA_rf_pca.csv", index=False)
print("Submission written: PROBA_rf_pca.csv")



Loading saved model...
Submission written: PROBA_rf_pca.csv


## Ensemble

In [ ]:
## RF + BOOST ENSEMBLE
import numpy as np
import pandas as pd
import os

# ---------- 1) Config ----------
coefs = [5, 2]  # RF, Boost
vote = False    # False => weighted sum, True => vote
name = f"ensemble_RF_BOOST_{coefs[0]}_{coefs[1]}"
if vote:
    name = f"ensembleVOTE_RF_BOOST_{coefs[0]}_{coefs[1]}"

os.makedirs("submissions", exist_ok=True)

# ---------- 2) Load probabilities from CSVs ----------
# Si tu as fait : PROBA_rf.csv et PROBA_boost.csv
prob_rf = pd.read_csv("PROBA_rf.csv").drop(columns="Id").to_numpy()
prob_boost = pd.read_csv("PROBA_boost.csv").drop(columns="Id").to_numpy()

# sécurité : mêmes dimensions
assert prob_rf.shape == prob_boost.shape, "RF et Boost n'ont pas la même shape"
n_classes = prob_rf.shape[1]

# ---------- 3) Predicted classes individuelles ----------
dfALL = pd.DataFrame({
    "rf": prob_rf.argmax(axis=1),
    "boost": prob_boost.argmax(axis=1),
})

# ---------- 4) Ensemble ----------
if vote:
    weights = {'rf': coefs[0], 'boost': coefs[1]}
    cols = []
    for col, w in weights.items():
        cols.extend([dfALL[col]] * w)
    weighted_df = pd.concat(cols, axis=1)
    dfALL[name] = weighted_df.mode(axis=1)[0].astype(int)
else:
    weighted_proba = coefs[0]*prob_rf + coefs[1]*prob_boost
    dfALL[name] = weighted_proba.argmax(axis=1)

# ---------- 5) Export ----------
result = dfALL[name].to_numpy()

pred_df = pd.DataFrame({
    "Id": np.arange(0, len(result)),
    "change_type": result
})

pred_df.to_csv(f"submissions/{name}.csv", index=False)
print(f"Submission saved as submissions/{name}.csv")


Submission saved as submissions/ensemble_RF_BOOST_5_2.csv


In [ ]:
## RF + BOOST + RF_PCA ENSEMBLE
import numpy as np
import pandas as pd
import os

# ---------- 1) Config ----------
coefs = [5, 2, 1]  # RF, Boost, RF2
vote = False    # False => weighted sum, True => vote
name = f"ensemble_RF_BOOST_PCA_{coefs[0]}_{coefs[1]}_{coefs[2]}"
if vote:
    name = f"ensembleVOTE_RF_BOOST_PCA_{coefs[0]}_{coefs[1]}_{coefs[2]}"

os.makedirs("submissions", exist_ok=True)

# ---------- 2) Load probabilities from CSVs ----------
# Si tu as fait : PROBA_rf.csv et PROBA_boost.csv
prob_rf = pd.read_csv("PROBA_rf.csv").drop(columns="Id").to_numpy()
prob_boost = pd.read_csv("PROBA_boost.csv").drop(columns="Id").to_numpy()
prob_rf2 = pd.read_csv("PROBA_rf_pca.csv").drop(columns="Id").to_numpy()

# sécurité : mêmes dimensions
assert prob_rf.shape == prob_boost.shape, "RF et Boost n'ont pas la même shape"
assert prob_rf.shape == prob_rf2.shape, "RF et RF2 n'ont pas la même shape"
n_classes = prob_rf.shape[1]

# ---------- 3) Predicted classes individuelles ----------
dfALL = pd.DataFrame({
    "rf": prob_rf.argmax(axis=1),
    "boost": prob_boost.argmax(axis=1),
    "rf2": prob_rf2.argmax(axis=1),
})

# ---------- 4) Ensemble ----------
if vote:
    weights = {'rf': coefs[0], 'boost': coefs[1], 'rf2': coefs[2]}
    cols = []
    for col, w in weights.items():
        cols.extend([dfALL[col]] * w)
    weighted_df = pd.concat(cols, axis=1)
    dfALL[name] = weighted_df.mode(axis=1)[0].astype(int)
else:
    weighted_proba = coefs[0]*prob_rf + coefs[1]*prob_boost + coefs[2]*prob_rf2
    dfALL[name] = weighted_proba.argmax(axis=1)

# ---------- 5) Export ----------
result = dfALL[name].to_numpy()

pred_df = pd.DataFrame({
    "Id": np.arange(0, len(result)),
    "change_type": result
})

pred_df.to_csv(f"submissions/{name}.csv", index=False)
print(f"Submission saved as submissions/{name}.csv")


Submission saved as submissions/ensemble_RF_BOOST_PCA_1_1.5_7.5.csv


In [ ]:
## RF + BOOST + RF_PCA + XGB ENSEMBLE
import numpy as np
import pandas as pd
import os

# ---------- 1) Config ----------
coefs = [5, 2, 1, 1]  # RF, Boost, RF2
vote = False    # False => weighted sum, True => vote
name = f"ensemble_RF_BOOST_PCA_XGB_{coefs[0]}_{coefs[1]}_{coefs[2]}_{coefs[3]}"
if vote:
    name = f"ensembleVOTE_RF_BOOST_PCA_XGB_{coefs[0]}_{coefs[1]}_{coefs[2]}_{coefs[3]}"

os.makedirs("submissions", exist_ok=True)

# ---------- 2) Load probabilities from CSVs ----------
# Si tu as fait : PROBA_rf.csv et PROBA_boost.csv
prob_rf = pd.read_csv("PROBA_rf.csv").drop(columns="Id").to_numpy()
prob_boost = pd.read_csv("PROBA_boost.csv").drop(columns="Id").to_numpy()
prob_rf2 = pd.read_csv("PROBA_rf_pca.csv").drop(columns="Id").to_numpy()
prob_xgb = pd.read_csv("PROBA_XGB.csv").drop(columns="Id").to_numpy()


# sécurité : mêmes dimensions
assert prob_rf.shape == prob_boost.shape, "RF et Boost n'ont pas la même shape"
assert prob_rf.shape == prob_rf2.shape, "RF et RF2 n'ont pas la même shape"
assert prob_rf.shape == prob_xgb.shape, "RF et XGB n'ont pas la même shape"
n_classes = prob_rf.shape[1]

# ---------- 3) Predicted classes individuelles ----------
dfALL = pd.DataFrame({
    "rf": prob_rf.argmax(axis=1),
    "boost": prob_boost.argmax(axis=1),
    "rf2": prob_rf2.argmax(axis=1),
    "xgb": prob_xgb.argmax(axis=1)
})

# ---------- 4) Ensemble ----------
if vote:
    weights = {'rf': coefs[0], 'boost': coefs[1], 'rf2': coefs[2], 'xgb': coefs[3]}
    cols = []
    for col, w in weights.items():
        cols.extend([dfALL[col]] * w)
    weighted_df = pd.concat(cols, axis=1)
    dfALL[name] = weighted_df.mode(axis=1)[0].astype(int)
else:
    weighted_proba = coefs[0]*prob_rf + coefs[1]*prob_boost + coefs[2]*prob_rf2 + coefs[3]*prob_xgb
    dfALL[name] = weighted_proba.argmax(axis=1)

# ---------- 5) Export ----------
result = dfALL[name].to_numpy()

pred_df = pd.DataFrame({
    "Id": np.arange(0, len(result)),
    "change_type": result
})

pred_df.to_csv(f"submissions/{name}.csv", index=False)
print(f"Submission saved as submissions/{name}.csv")


Submission saved as submissions/ensemble_RF_BOOST_PCA_XGB_5_2_1_1.csv
